In [1]:
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(tidyr))
suppressPackageStartupMessages(library(tibble))
suppressPackageStartupMessages(library(stringr))
library(parallel)
library(DESeq2)
library(hise)
source('/home//workspace/pediatric_IH/jupyter/IHA-Figure/helper_function/helper_function_IHA.r')

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    Filter, Find, Map, Position, Reduce, anyDuplicated, aperm, append,
    as.data.frame, basename, cbind, colnames, dirname, do.call,
    duplicated, eval, evalq, get, grep, grepl, intersect, is.unsorted,
    lapply, mapply, match, mget, order, paste, pmax, pmax.int, pmin,
    pmin.int, rank, rbind, rownames, sapply, saveRDS, setdiff, table,
    tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:tidyr’:

    expand


The following objects are masked from ‘package:dplyr’:

    first, rename


The following object is masked f

# Read Meta Data

In [2]:
#meta_data=read.csv("/home/workspace/pediatric_IH/jupyter/certpro_up1/BRI_Analysis/scRNA_meta_data-2024-05-09.csv")
meta_data=read.csv("/home/workspace/pediatric_IH/jupyter/preprocessing_output/UP1_218scRNAsamp_noDuplicates.csv")

In [3]:
colnames(meta_data)
#meta_data$cmv_status

[1] "X"                                   "Unnamed..0.1"                       
 [3] "Unnamed..0"                          "lastUpdated"                        
 [5] "sample.id"                           "sample.bridgingControl"             
 [7] "sample.sampleKitGuid"                "sample.visitName"                   
 [9] "sample.visitDetails"                 "sample.drawDate"                    
[11] "sample.daysSinceFirstVisit"          "sample.diseaseStatesRecordedAtVisit"
[13] "file.id"                             "file.name"                          
[15] "file.batchID"                        "file.panel"                         
[17] "file.pool"                           "file.fileType"                      
[19] "file.majorVersion"                   "subject.id"                         
[21] "subject.biologicalSex"               "subject.birthYear"                  
[23] "subject.ethnicity"                   "subject.partnerCode"                
[25] "subject.race"                        "subject.subjectGuid"                
[27] "cohort.cohortGuid"                   "file.userTags.details"              
[29] "file.userTags.group"                 "file.userTags.name"                 
[31] "file.userTags.origin"                "file.userTags.other"                
[33] "file.userTags.version"               "specimens.specimenType"             
[35] "specimens.specimenGuid"              "cmv_status"                         
[37] "relabeled_visit"

In [4]:
df<-as.data.frame(table(meta_data$subject.subjectGuid))

In [5]:
df$sub<-df$Var1

In [6]:
df$cohort<-substr(as.character(df$Var1),1,3)

In [7]:
as.character(df$Var1)

[1] "UP1001" "UP1002" "UP1003" "UP1004" "UP1005" "UP1006" "UP1007" "UP1008"
 [9] "UP1009" "UP1010" "UP1011" "UP1012" "UP1013" "UP1014" "UP1015" "UP1016"
[17] "UP1017" "UP1018" "UP1019" "UP1020" "UP1021" "UP1022" "UP1023" "UP1024"
[25] "UP1025" "UP1026" "UP1027" "UP1028" "UP1031" "UP1032" "UP1033" "UP1034"
[33] "UP1035" "UP1036" "UP1037" "UP1038" "UP1039" "UP1041" "UP1042" "UP1043"
[41] "UP1044" "UP1045" "UP1047" "UP1048" "UP1049" "UP1050"

In [8]:
#meta_data=meta_data %>% filter(cohort.cohortGuid == 'BR2')

In [8]:
df
table(meta_data$sample.visitName)

Var1,Freq,sub,cohort
<fct>,<int>,<fct>,<chr>
UP1001,10,UP1001,UP1
UP1002,2,UP1002,UP1
UP1003,10,UP1003,UP1
UP1004,10,UP1004,UP1
UP1005,7,UP1005,UP1
UP1006,10,UP1006,UP1
UP1007,8,UP1007,UP1
UP1008,6,UP1008,UP1
UP1009,1,UP1009,UP1



             COVID-19 Visit 1              COVID-19 Visit 3 
                            2                            10 
             COVID-19 Visit 6        Flu Year 1 Day 180-360 
                           12                            12 
             Flu Year 1 Day 7             Flu Year 1 Day 90 
                           39                            27 
Flu Year 1 Pre-Vac 7-12 Weeks        Flu Year 1 Stand-Alone 
                           46                             1 
       Flu Year 2 Day 180-360              Flu Year 2 Day 7 
                           16                            18 
            Flu Year 2 Day 90 Flu Year 2 Pre-Vac 7-12 Weeks 
                           16                            19 

In [31]:
#trying to run on only the subjects that have Year 1 and 2 samples
subjects_with_both_years <- meta_data %>%
  group_by(subject.subjectGuid) %>%
  filter(all(c('Flu Year 1 Pre-Vac 7-12 Weeks', 'Flu Year 1 Day 7', 'Flu Year 2 Pre-Vac 7-12 Weeks','Flu Year 2 Day 7') %in% sample.visitName)) %>%
  ungroup()

meta_data <- meta_data  %>% filter(meta_data$subject.subjectGuid %in% subjects_with_both_years$subject.subjectGuid )

# Perform Deseq2 on Year 1 Day 0 samples

In [9]:
filtered_gene_set<-read.csv("/home/workspace/pediatric_IH/jupyter/Analysis/DEG/Flu_Y2_D0_D7/UP1_filtered_gene_Y2_D0_D90_clean.csv")
cell_list = read.csv('/home/workspace/pediatric_IH/jupyter/Analysis/DEG/cells.csv')

In [10]:
df_list <-list()

In [11]:
#meta_data_subset=meta_data %>% filter(sample.visitName  %in% c("Flu Year 1 Pre-Vac 7-12 Weeks", "Flu Year 1 Day 7"))

aggregated_count_file_list<- paste0('/home/workspace/pediatric_IH/jupyter/pseudobulk_output/Aggregated_Raw_Expression_of_Celltypes_by_celltype_AIFI_L3/',cell_list$celltype,'.csv')
#paste0("/home/jupyter/BRI_Figures_Final_V2/Dataset/scRNA/BRI/Aggregated_Count/Aggregated_Raw_Expression_of_Celltypes_by_Sample_AIFI_L3/",meta_data_subset$pbmc_sample_id,".csv")



In [12]:
df_list<-read_pseudobulk_expression(aggregated_count_file_list)

[1] "Total reading time: 24.5290000000095 seconds"
[1] "The length of the list matches the length of the input path."


In [13]:
df_list[[1]]

,ILC:KT00124,ILC:KT00127,ILC:KT00172,ILC:KT00173,ILC:KT00175,ILC:KT00180,ILC:KT00182,ILC:KT00190,ILC:KT00192,ILC:KT00193,⋯,ILC:KT03991,ILC:KT03993,ILC:KT03994,ILC:KT03995,ILC:KT04345,ILC:KT04348,ILC:KT04349,ILC:KT04351,ILC:KT04352,ILC:KT04356
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MIR1302-2HG,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
FAM138A,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
OR4F5,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.1,0,0,0,0,0,1,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.3,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.2,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.4,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL732372.1,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
OR4F29,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [13]:
celltypes <- unlist(lapply(df_list, function(x){
    gsub(":.*","", colnames(x)[1])
}))
names(df_list) <- celltypes

### Deseq analysis for only one sample

In [79]:
exp_matrix<-df_list[[69]]
exp_df <- as.data.frame(exp_matrix)
exp_df

,ASDC:KT00124,ASDC:KT00127,ASDC:KT00172,ASDC:KT00173,ASDC:KT00175,ASDC:KT00180,ASDC:KT00182,ASDC:KT00190,ASDC:KT00192,ASDC:KT00193,⋯,ASDC:KT03991,ASDC:KT03993,ASDC:KT03994,ASDC:KT03995,ASDC:KT04345,ASDC:KT04348,ASDC:KT04349,ASDC:KT04351,ASDC:KT04352,ASDC:KT04356
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MIR1302-2HG,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
FAM138A,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
OR4F5,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.1,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.3,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.2,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.4,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL732372.1,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
OR4F29,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [80]:
#rename colnames to only be the kitID
colnames(exp_df) <- gsub(".*:", "", colnames(exp_df))
head(exp_df)

,KT00124,KT00127,KT00172,KT00173,KT00175,KT00180,KT00182,KT00190,KT00192,KT00193,⋯,KT03991,KT03993,KT03994,KT03995,KT04345,KT04348,KT04349,KT04351,KT04352,KT04356
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MIR1302-2HG,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
FAM138A,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
OR4F5,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.1,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.3,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
AL627309.2,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [81]:
meta_data_subset=meta_data %>% filter(sample.sampleKitGuid	 %in% colnames(exp_df),
                                      sample.visitName %in% c('Flu Year 1 Pre-Vac 7-12 Weeks',
                                                              'Flu Year 1 Day 7'))

In [82]:
dim(meta_data_subset)
dim(exp_df)

[1] 83 33

[1] 33538   214

In [83]:
#looking at the metadata
unique(meta_data_subset$sample.visitName)

[1] "Flu Year 1 Pre-Vac 7-12 Weeks" "Flu Year 1 Day 7"

In [84]:
rownames(meta_data_subset)<-meta_data_subset$sample.sampleKitGuid
#head(meta_data_subset)
dim(meta_data_subset)

[1] 83 33

In [86]:
#Get only 1 cell from df_list
celltype<-celltypes[[69]]
celltype

[1] "ASDC"

In [87]:
dim(exp_df)

[1] 33538   214

In [88]:
length(unique(rownames(meta_data_subset)))
length(unique(colnames(exp_df)))

[1] 83

[1] 214

In [89]:
#now subset exp_df so that its colnames matches the rows on meta data
exp_df <- exp_df[, rownames(meta_data_subset)]
dim(exp_df)

[1] 33538    83

In [92]:
rownames(meta_data_subset)
colnames(exp_df)

[1] "KT00809" "KT00813" "KT00815" "KT00824" "KT00828" "KT00852" "KT00897"
 [8] "KT00172" "KT00173" "KT00197" "KT00807" "KT00811" "KT00847" "KT00848"
[15] "KT00886" "KT00841" "KT00842" "KT00853" "KT00190" "KT00192" "KT00124"
[22] "KT00175" "KT00180" "KT00193" "KT00182" "KT00127" "KT00199" "KT00810"
[29] "KT00921" "KT00925" "KT00924" "KT00937" "KT00961" "KT00958" "KT00913"
[36] "KT00927" "KT00914" "KT00931" "KT00928" "KT02390" "KT00884" "KT00889"
[43] "KT00910" "KT00833" "KT00892" "KT00907" "KT00825" "KT00879" "KT00882"
[50] "KT03254" "KT03255" "KT03965" "KT03250" "KT03223" "KT02372" "KT02939"
[57] "KT02391" "KT02901" "KT02392" "KT03950" "KT03948" "KT03993" "KT03963"
[64] "KT03020" "KT03036" "KT03253" "KT02415" "KT03974" "KT03951" "KT03984"
[71] "KT04348" "KT03958" "KT03994" "KT03969" "KT03989" "KT03988" "KT03985"
[78] "KT03979" "KT03976" "KT03970" "KT04356" "KT03982" "KT03986"

[1] "KT00809" "KT00813" "KT00815" "KT00824" "KT00828" "KT00852" "KT00897"
 [8] "KT00172" "KT00173" "KT00197" "KT00807" "KT00811" "KT00847" "KT00848"
[15] "KT00886" "KT00841" "KT00842" "KT00853" "KT00190" "KT00192" "KT00124"
[22] "KT00175" "KT00180" "KT00193" "KT00182" "KT00127" "KT00199" "KT00810"
[29] "KT00921" "KT00925" "KT00924" "KT00937" "KT00961" "KT00958" "KT00913"
[36] "KT00927" "KT00914" "KT00931" "KT00928" "KT02390" "KT00884" "KT00889"
[43] "KT00910" "KT00833" "KT00892" "KT00907" "KT00825" "KT00879" "KT00882"
[50] "KT03254" "KT03255" "KT03965" "KT03250" "KT03223" "KT02372" "KT02939"
[57] "KT02391" "KT02901" "KT02392" "KT03950" "KT03948" "KT03993" "KT03963"
[64] "KT03020" "KT03036" "KT03253" "KT02415" "KT03974" "KT03951" "KT03984"
[71] "KT04348" "KT03958" "KT03994" "KT03969" "KT03989" "KT03988" "KT03985"
[78] "KT03979" "KT03976" "KT03970" "KT04356" "KT03982" "KT03986"

In [91]:
meta_data_subset <-meta_data_subset[match(colnames(exp_df),rownames(meta_data_subset)),] #order
#meta_data_subset

In [95]:
length(filtered_gene_set_filtered)

[1] 6962

In [94]:
#Now working on the filtered_gene_set
filtered_gene_set_filtered<-filtered_gene_set%>% filter(AIFI_L3==celltype)%>% select(gene) %>%pull()

In [99]:
dim(exp_df)

[1] 6962   83

In [97]:
#filtered_gene_set_filtered <- filtered_gene_set_filtered[filtered_gene_set_filtered %in% rownames(exp_df)]
length(filtered_gene_set_filtered)

[1] 6962

In [98]:
exp_df <- exp_df[filtered_gene_set_filtered,]

In [100]:
res=deseq2_analysis(exp_df,
                 meta_data=meta_data_subset,
                 filtered_gene_set=filtered_gene_set_filtered,#rownames(exp_matrix),
                 #add visit to formula
                 formula= ~ sample.visitName,#+subject.race,#subject.biologicalSex+subject.race,
                 comparisons=list(c('sample.visitName', "Flu Year 1 Pre-Vac 7-12 Weeks", "Flu Year 1 Day 7")),
                                  #c("subject.race", "Caucasian", "African American", "Asian")),
                  celltype=celltype)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is reco

In [101]:
res

baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,Direction,gene,celltype
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
2.4798455,-0.7957567,0.2433718,-3.269717,0.0010765528,0.7030648,sample.visitName,HigherInFlu Year 1 Day 7,PRDM2,ASDC
1.9231031,-0.8830772,0.2631174,-3.356210,0.0007901861,0.7030648,sample.visitName,HigherInFlu Year 1 Day 7,EVI5,ASDC
6.8855911,0.4660892,0.1297433,3.592395,0.0003276527,0.7030648,sample.visitName,HigherInFlu Year 1 Pre-Vac 7-12 Weeks,TSTD1,ASDC
1.8419689,-0.8461319,0.2608035,-3.244327,0.0011772846,0.7030648,sample.visitName,HigherInFlu Year 1 Day 7,CASP8,ASDC
0.8105763,-1.2494042,0.3618816,-3.452522,0.0005553727,0.7030648,sample.visitName,HigherInFlu Year 1 Day 7,FAM193A,ASDC
1.1900800,-0.9256112,0.2857731,-3.238973,0.0011996111,0.7030648,sample.visitName,HigherInFlu Year 1 Day 7,RAB6A,ASDC
10.2733591,-0.4539311,0.1274155,-3.562605,0.0003671927,0.7030648,sample.visitName,HigherInFlu Year 1 Day 7,AKAP13,ASDC
7.8632250,0.4594670,0.1350958,3.401046,0.0006712849,0.7030648,sample.visitName,HigherInFlu Year 1 Pre-Vac 7-12 Weeks,CKLF,ASDC
9.4487901,-0.3679180,0.1113851,-3.303118,0.0009561630,0.7030648,sample.visitName,HigherInFlu Year 1 Day 7,EVI2B,ASDC


### deseq loop

In [39]:
meta_data %>% filter( sample.visitName %in% c('Flu Year 2 Pre-Vac 7-12 Weeks','Flu Year 2 Day 7'))

X,Unnamed..0.1,Unnamed..0,lastUpdated,sample.id,sample.bridgingControl,sample.sampleKitGuid,sample.visitName,sample.visitDetails,sample.drawDate,⋯,file.userTags.details,file.userTags.group,file.userTags.name,file.userTags.origin,file.userTags.other,file.userTags.version,specimens.specimenType,specimens.specimenGuid,cmv_status,relabeled_visit
<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<chr>
47,47,48,2024-08-17T13:57:19.462Z,45440bf3-f13a-40a6-a56c-56fdbb66b332,False,KT02374,Flu Year 2 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2021-08-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,Plasma,PL02374-020,Negative,Flu Year 2 Pre-Vac 7-12 Weeks
48,48,49,2024-08-17T13:57:19.462Z,9f091af2-be70-4e87-9456-f41ac1ff73af,False,KT02379,Flu Year 2 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2021-08-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,Plasma,PL02379-002,Positive,Flu Year 2 Pre-Vac 7-12 Weeks
51,51,52,2024-08-17T13:57:19.462Z,cbffe35e-b428-42d3-9127-6c841347bb0e,False,KT02380,Flu Year 2 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2021-08-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,PBMC,PB02380-011,Positive,Flu Year 2 Pre-Vac 7-12 Weeks
84,84,85,2024-08-17T13:57:19.462Z,323d20ee-e555-4664-94f4-79ae84a86181,False,KT02416,Flu Year 2 Day 7,N/A - Flu-Series Timepoint Only,2021-09-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,Plasma,PL02416-003,Negative,Flu Year 2 Day 7
85,85,86,2024-08-17T13:57:19.462Z,3eccd3e2-d1de-4a10-9edb-b9e634197f77,False,KT02394,Flu Year 2 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2021-09-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,PBMC,PB02394-002,Negative,Flu Year 2 Pre-Vac 7-12 Weeks
86,86,87,2024-08-17T13:57:19.462Z,e8489fd4-02a6-4013-8fab-e4acdeaa8d9d,False,KT02387,Flu Year 2 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2021-08-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,PBMC,PB02387-001,Negative,Flu Year 2 Pre-Vac 7-12 Weeks
87,87,88,2024-08-17T13:57:19.462Z,9f2088a3-6c91-47ae-8d26-4dde9bc05072,False,KT02410,Flu Year 2 Day 7,N/A - Flu-Series Timepoint Only,2021-09-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,PBMC,PB02410-009,Negative,Flu Year 2 Day 7
88,88,89,2024-08-17T13:57:19.462Z,da5f10f0-facd-4ac2-baaa-e86c390e1252,False,KT02385,Flu Year 2 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2021-08-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,Plasma,PL02385-016,Negative,Flu Year 2 Pre-Vac 7-12 Weeks
89,89,90,2024-08-17T13:57:19.462Z,4f30d6d0-bc17-4041-affc-37ae3a2933b8,False,KT02381,Flu Year 2 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2021-08-01T00:00:00Z,⋯,NA,NA,NA,NA,NA,NA,Plasma,PL02381-010,Negative,Flu Year 2 Pre-Vac 7-12 Weeks


In [20]:
res_list <- lapply(celltypes, function(celltype){
    print(celltype)
    celltype_list <- lapply(df_list, function(df) { df <- df[, grep(celltype, names(df), fixed = TRUE),drop=FALSE] })
    exp_matrix<-do.call(cbind,celltype_list)
    colnames(exp_matrix)<-gsub(".*:","",colnames(exp_matrix))
    #print(exp_matrix)

    meta_data_subset=meta_data %>% filter(sample.sampleKitGuid	 %in% colnames(exp_matrix),
                                       sample.visitName %in% c('Flu Year 2 Pre-Vac 7-12 Weeks',
                                                               'Flu Year 2 Day 90'))
    rownames(meta_data_subset)<-meta_data_subset$sample.sampleKitGuid
    print(nrow(meta_data_subset))
    
    exp_matrix <- exp_matrix[, rownames(meta_data_subset)] #subsetting exp_matrix so that column matches rownames(meta_data_subset)  
    # print(dim(exp_matrix))
    # print(dim(meta_data_subset))
    filtered_gene_set_filtered<-filtered_gene_set%>% filter(AIFI_L3==celltype)%>% select(gene) %>%pull()
    res=deseq2_analysis(exp_matrix,
                     meta_data=meta_data_subset,
                     filtered_gene_set=filtered_gene_set_filtered,
                     formula= ~ subject.subjectGuid+sample.visitName,   #+cmv_status+subject.biologicalSex,
                     comparisons=list(c('sample.visitName', "Flu Year 2 Day 90", "Flu Year 2 Pre-Vac 7-12 Weeks")),
                                      # c('cmv_status', 'Positive','Negative'),
                                      # c('subject.biologicalSex', 'Male','Female')),
                     celltype=celltype)
    return(res)
})

[1] "ILC"
[1] 35


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is reco

[1] "GZMK- CD27+ EM CD8 T cell"
[1] 35


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is reco

[1] "KLRB1+ memory CD4 Treg"
[1] 35


...

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is reco

[1] "ASDC"
[1] 34


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is reco

[1] "CLP cell"
[1] 29


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is reco

In [18]:
res_list <- mclapply(celltypes, function(celltype){
    print(celltype)
    celltype_list <- lapply(df_list, function(df) { df <- df[, grep(celltype, names(df), fixed = TRUE),drop=FALSE] })
    exp_matrix<-do.call(cbind,celltype_list)
    colnames(exp_matrix)<-gsub(".*:","",colnames(exp_matrix))
    #print(exp_matrix)

    meta_data_subset=meta_data %>% filter(sample.sampleKitGuid	 %in% colnames(exp_matrix),
                                       sample.visitName %in% c('Flu Year 2 Pre-Vac 7-12 Weeks',
                                                               'Flu Year 2 Day 90'))
    rownames(meta_data_subset)<-meta_data_subset$sample.sampleKitGuid
    
    exp_matrix <- exp_matrix[, rownames(meta_data_subset)] #subsetting exp_matrix so that column matches rownames(meta_data_subset)  
    # print(dim(exp_matrix))
    # print(dim(meta_data_subset))
    filtered_gene_set_filtered<-filtered_gene_set%>% filter(AIFI_L3==celltype)%>% select(gene) %>%pull()
    res=deseq2_analysis(exp_matrix,
                     meta_data=meta_data_subset,
                     filtered_gene_set=filtered_gene_set_filtered,
                     formula= ~ subject.subjectGuid+sample.visitName,   #+cmv_status+subject.biologicalSex,
                     comparisons=list(c('sample.visitName', "Flu Year 2 Day 90", "Flu Year 2 Pre-Vac 7-12 Weeks")),
                                      # c('cmv_status', 'Positive','Negative'),
                                      # c('subject.biologicalSex', 'Male','Female')),
                     celltype=celltype)
    return(res)
},mc.cores=60)

Warning message in mclapply(celltypes, function(celltype) {:
“scheduled cores 57, 56, 53, 52, 51, 50, 49, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 9, 1, 12, 59, 60, 15, 2, 7, 3, 11, 18, 16, 24, 17, 13, 14, 21, 27, 28, 31, 22, 5, 10, 4, 20, 30, 8, 26, 23, 19, 6, 25, 29 encountered errors in user code, all values of the jobs will be affected”


In [19]:
res_list

[[1]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[2]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[3]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[4]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[5]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[6]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[7]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[8]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[9]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[10]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[11]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[12]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[13]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] "try-error"

[[14]]
[1] "fatal error in wrapper code"
attr(,"class")
[1] 

In [21]:
res_all<-do.call(rbind,res_list)

In [25]:
res_all

baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,Direction,gene,celltype
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
3.298319,0.107949275,0.44477867,0.24270335,0.80823521,0.9986237,sample.visitName,HigherInFlu Year 2 Day 90,ISG15,ILC
3.657115,-0.125371511,0.40205015,-0.31183052,0.75516932,0.9986237,sample.visitName,HigherInFlu Year 2 Day 7,TTLL10,ILC
26.286094,-0.071188685,0.17063776,-0.41719186,0.67653808,0.9986237,sample.visitName,HigherInFlu Year 2 Day 7,TNFRSF18,ILC
21.335181,0.267740850,0.17457115,1.53370613,0.12510196,0.9986237,sample.visitName,HigherInFlu Year 2 Day 90,TNFRSF4,ILC
3.529923,-0.212888857,0.37938235,-0.56114591,0.57469807,0.9986237,sample.visitName,HigherInFlu Year 2 Day 7,SDF4,ILC
2.715087,0.329381690,0.47014452,0.70059668,0.48355475,0.9986237,sample.visitName,HigherInFlu Year 2 Day 90,UBE2J2,ILC
1.626662,0.539080172,0.63053026,0.85496320,0.39257152,0.9986237,sample.visitName,HigherInFlu Year 2 Day 90,INTS11,ILC
5.371769,-0.287098333,0.32736883,-0.87698739,0.38049349,0.9986237,sample.visitName,HigherInFlu Year 2 Day 7,AURKAIP1,ILC
1.806097,-0.316148121,0.61783345,-0.51170444,0.60885788,0.9986237,sample.visitName,HigherInFlu Year 2 Day 7,CCNL2,ILC


In [22]:
#write.csv(res_all, 'BR2_deseq_y2_d7_d0_subj_clean.csv')
write.csv(res_all, 'UP1_deseq_Y2D90D0.csv')